In [1]:
!pip install spacy sentence-transformers scikit-learn networkx
!python -m spacy download en_core_web_lg

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
qwen-tts 0.1.1 requires sox, which is not installed.
qwen-tts 0.1.1 requires accelerate==1.12.0, but you have accelerate 1.2.1 which is incompatible.
qwen-tts 0.1.1 requires transformers==4.57.3, but you have transformers 4.48.0 which is incompatible.

[notice] A new release of pip is available: 24.3.1 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip



  Using cached sentence_transformers-5.2.3-py3-none-any.whl.metadata (16 kB)
   ---------------------------------------- 0.0/566.4 kB ? eta -:--:--
   ---------------------------------------- 0.0/566.4 kB ? eta -:--:--
   ---------------------------------------- 0.0/566.4 kB ? eta -:--:--
   ---------------------------------------- 0.0/566.4 kB ? eta -:--:--
   ------------------ --------------------- 262.1/566.4 kB ? eta -:--:--
   ------------------ --------------------- 262.1/566.4 kB ? eta -:--:--
   ----------------------------------- -- 524.3/566.4 kB 430.4 kB/s eta 0:00:01
   -------------------------------------- 566.4/566.4 kB 463.8 kB/s eta 0:00:00
  Attempting uninstall: huggingface-hub
    Found existing installation: huggingface_hub 1.5.0
    Uninstalling huggingface_hub-1.5.0:
      Successfully uninstalled huggingface_hub-1.5.0
^C


### STEP 1 — Extract all PERSON entities from the books using spaCy NER

Input:  books/*.txt  (plain text files of each ASOIAF book)  <br>
Output: data/raw_persons.json  — { "name": count, ... }

In [4]:
import json
import os
from collections import Counter
from pathlib import Path

import spacy

# ── Config ────────────────────────────────────────────────────────────────────
BOOKS_DIR   = Path("books")          # folder containing .txt files
OUTPUT_DIR  = Path("data")
OUTPUT_FILE = OUTPUT_DIR / "raw_persons.json"
CHUNK_SIZE  = 100_000                # characters per spaCy chunk (memory limit)
MIN_RAW_MENTIONS = 3                 # drop names seen fewer than N times total
# ─────────────────────────────────────────────────────────────────────────────

def load_books(books_dir: Path) -> dict[str, str]:
    """Load every .txt file in the books directory."""
    texts = {}
    for path in sorted(books_dir.glob("*.txt")):
        print(f"  Loading {path.name} …")
        texts[path.stem] = path.read_text(encoding="utf-8", errors="replace")
    return texts


def extract_persons_from_text(text: str, nlp) -> Counter:
    """
    Run spaCy NER over `text` in chunks.
    Returns a Counter of raw entity strings labelled PERSON.
    """
    persons: Counter = Counter()
    total = len(text)

    for start in range(0, total, CHUNK_SIZE):
        chunk = text[start : start + CHUNK_SIZE]
        doc = nlp(chunk)
        for ent in doc.ents:
            if ent.label_ == "PERSON":
                name = ent.text.strip()
                # Basic sanity: at least 2 chars, not purely punctuation
                if len(name) >= 2 and any(c.isalpha() for c in name):
                    persons[name] += 1

        pct = min(start + CHUNK_SIZE, total) / total * 100
        print(f"    {pct:5.1f}%", end="\r", flush=True)

    print()
    return persons


def main():
    OUTPUT_DIR.mkdir(exist_ok=True)

    print("Loading spaCy model (en_core_web_lg) …")
    nlp = spacy.load("en_core_web_lg")
    # Disable components we don't need for speed
    nlp.select_pipes(enable=["tok2vec", "ner"])

    print(f"\nScanning books in '{BOOKS_DIR}/' …")
    books = load_books(BOOKS_DIR)
    if not books:
        raise FileNotFoundError(
            f"No .txt files found in '{BOOKS_DIR}/'. "
            "Put your book files there and re-run."
        )

    all_persons: Counter = Counter()

    for book_name, text in books.items():
        print(f"\n[{book_name}]  ({len(text):,} chars)")
        persons = extract_persons_from_text(text, nlp)
        print(f"  → {len(persons)} unique raw names, "
              f"{sum(persons.values())} total mentions")
        all_persons.update(persons)

    # Filter by minimum mentions
    filtered = {
        name: count
        for name, count in all_persons.items()
        if count >= MIN_RAW_MENTIONS
    }

    print(f"\n{'─'*50}")
    print(f"Total unique raw names (>= {MIN_RAW_MENTIONS} mentions): {len(filtered)}")
    print(f"\nTop 30 most mentioned:")
    for name, count in Counter(filtered).most_common(30):
        print(f"  {count:5d}  {name}")

    with open(OUTPUT_FILE, "w", encoding="utf-8") as f:
        json.dump(filtered, f, indent=2, ensure_ascii=False)

    print(f"\n✓ Saved to {OUTPUT_FILE}")

main()

Loading spaCy model (en_core_web_lg) …

Scanning books in 'books/' …
  Loading GOT1.txt …
  Loading GOT2.txt …
  Loading GOT3.txt …
  Loading GOT4.txt …
  Loading GOT5.txt …

[GOT1]  (1,591,192 chars)
    100.0%
  → 802 unique raw names, 10524 total mentions

[GOT2]  (1,736,138 chars)
    100.0%
  → 1190 unique raw names, 10607 total mentions

[GOT3]  (2,250,034 chars)
    100.0%
  → 1620 unique raw names, 15138 total mentions

[GOT4]  (1,601,992 chars)
    100.0%
  → 1468 unique raw names, 9873 total mentions

[GOT5]  (2,258,559 chars)
    100.0%
  → 1660 unique raw names, 13265 total mentions

──────────────────────────────────────────────────
Total unique raw names (>= 3 mentions): 1740

Top 30 most mentioned:
   2376  Jon
   2305  Tyrion
   1376  Arya
   1352  Jaime
   1307  Dany
    967  Robb
    961  Grace
    932  Stannis
    925  Robert
    921  Catelyn
    921  Sam
    849  Ned
    846  Cersei
    833  Winterfell
    770  Joffrey
    694  Brienne
    620  Theon
    565  Davos


### STEP 2 — Cluster raw person names into canonical characters using embeddings.

Strategy: sentence-transformers embeddings + Agglomerative Clustering. <br>
  <br>
Input:  data/raw_persons.json        (from step 1)  <br>
Output: data/canonical_characters.json  — list of character dicts  <br>
        data/alias_map.json             — { alias: canonical_name }  <br>

In [2]:
import json
import re
from collections import defaultdict
from pathlib import Path

import numpy as np
from sentence_transformers import SentenceTransformer
from sklearn.cluster import AgglomerativeClustering
from sklearn.metrics.pairwise import cosine_distances

# ── Config ────────────────────────────────────────────────────────────────────
INPUT_FILE       = Path("data/raw_persons.json")
OUTPUT_CHARS     = Path("data/canonical_characters.json")
OUTPUT_ALIAS_MAP = Path("data/alias_map.json")

# Embedding model — small but good for names
EMBED_MODEL      = "all-MiniLM-L6-v2"

# Agglomerative clustering threshold (cosine distance 0–2).
# Lower = stricter (fewer merges). Tune this if clusters are too broad/narrow.
DISTANCE_THRESHOLD = 0.35

# After clustering, drop characters mentioned fewer than N times total
MIN_TOTAL_MENTIONS = 10

# Noise prefixes — names starting with these are likely titles, not characters
NOISE_PREFIXES = {
    "the", "my", "your", "his", "her", "our", "their",
    "a", "an", "this", "that", "old", "young", "poor",
    "good", "great", "ser",   # 'ser' alone without a surname is kept later
}

# Noise single tokens (case-insensitive)
NOISE_TOKENS = {
    "lord", "lady", "king", "queen", "prince", "princess",
    "maester", "septon", "septa", "knight", "guard",
    "god", "gods", "stranger", "smith", "maiden", "mother", "father",
    "brother", "sister", "son", "daughter",
}
# ─────────────────────────────────────────────────────────────────────────────


def load_raw_persons(path: Path) -> dict[str, int]:
    with open(path, encoding="utf-8") as f:
        return json.load(f)


def is_noise(name: str) -> bool:
    """Return True if the name looks like a title or generic phrase."""
    tokens = name.lower().split()
    if not tokens:
        return True
    if tokens[0] in NOISE_PREFIXES:
        return True
    # Single generic word
    if len(tokens) == 1 and tokens[0] in NOISE_TOKENS:
        return True
    # Contains digits
    if any(c.isdigit() for c in name):
        return True
    # Very short single-token non-capitalised
    if len(tokens) == 1 and len(name) <= 2:
        return True
    return False


def clean_name(name: str) -> str:
    """Strip extra whitespace and quotation marks."""
    name = re.sub(r"[\"''\u2018\u2019\u201c\u201d]", "", name)
    return " ".join(name.split())


def embed_names(names: list[str], model_name: str) -> np.ndarray:
    """Embed a list of names using sentence-transformers."""
    print(f"  Loading embedding model '{model_name}' …")
    model = SentenceTransformer(model_name)
    print(f"  Embedding {len(names)} names …")
    embeddings = model.encode(names, show_progress_bar=True, batch_size=128)
    return embeddings


def cluster_embeddings(
    embeddings: np.ndarray,
    threshold: float,
) -> np.ndarray:
    """
    Agglomerative clustering with cosine distance.
    Returns an array of cluster labels (one per name).
    """
    print(f"  Clustering (threshold={threshold}) …")
    dist_matrix = cosine_distances(embeddings)

    clustering = AgglomerativeClustering(
        n_clusters=None,
        distance_threshold=threshold,
        metric="precomputed",
        linkage="average",
    )
    labels = clustering.fit_predict(dist_matrix)
    n_clusters = labels.max() + 1
    print(f"  → {n_clusters} clusters found")
    return labels


def build_canonical_map(
    names: list[str],
    labels: np.ndarray,
    counts: dict[str, int],
) -> tuple[dict[str, str], list[dict]]:
    """
    For each cluster, pick the canonical name = alias with most mentions.
    Returns:
      alias_map   : { any_alias → canonical_name }
      characters  : list of { id, name, aliases, mentions }
    """
    clusters: dict[int, list[str]] = defaultdict(list)
    for name, label in zip(names, labels):
        clusters[int(label)].append(name)

    alias_map: dict[str, str] = {}
    characters: list[dict] = []

    for label, aliases in clusters.items():
        # Canonical = highest mention count among aliases
        canonical = max(aliases, key=lambda n: counts.get(n, 0))
        total_mentions = sum(counts.get(a, 0) for a in aliases)

        for alias in aliases:
            alias_map[alias] = canonical

        characters.append({
            "id": canonical.lower().replace(" ", "_"),
            "name": canonical,
            "aliases": sorted(set(aliases) - {canonical}),
            "mentions": total_mentions,
        })

    # Sort by mention count descending
    characters.sort(key=lambda c: c["mentions"], reverse=True)
    return alias_map, characters


def main():
    print("── Step 2: Alias Clustering via Embeddings ──\n")

    raw = load_raw_persons(INPUT_FILE)
    print(f"Loaded {len(raw)} raw names from {INPUT_FILE}")

    # ── 1. Clean & filter noise ───────────────────────────────────────────────
    cleaned: dict[str, int] = {}
    for name, count in raw.items():
        name = clean_name(name)
        if not is_noise(name) and name:
            # Accumulate counts (cleaning may merge duplicates)
            cleaned[name] = cleaned.get(name, 0) + count

    print(f"After noise filtering: {len(cleaned)} names")

    # ── 2. Embed ──────────────────────────────────────────────────────────────
    names = list(cleaned.keys())
    embeddings = embed_names(names, EMBED_MODEL)

    # ── 3. Cluster ────────────────────────────────────────────────────────────
    labels = cluster_embeddings(embeddings, DISTANCE_THRESHOLD)

    # ── 4. Build canonical map ────────────────────────────────────────────────
    alias_map, characters = build_canonical_map(names, labels, cleaned)

    # ── 5. Filter by total mentions ───────────────────────────────────────────
    characters = [c for c in characters if c["mentions"] >= MIN_TOTAL_MENTIONS]
    # Rebuild alias_map to only include characters that passed the filter
    surviving = {c["name"] for c in characters}
    alias_map = {a: c for a, c in alias_map.items() if c in surviving}

    # ── 6. Report ─────────────────────────────────────────────────────────────
    print(f"\n{'─'*55}")
    print(f"Characters after filtering (>= {MIN_TOTAL_MENTIONS} mentions): {len(characters)}")
    print(f"\nTop 30 characters:")
    for char in characters[:30]:
        aliases_str = ", ".join(char["aliases"][:4])
        if len(char["aliases"]) > 4:
            aliases_str += f" (+{len(char['aliases'])-4} more)"
        print(f"  {char['mentions']:5d}  {char['name']:<28}  [{aliases_str}]")

    # ── 7. Save ───────────────────────────────────────────────────────────────
    Path("data").mkdir(exist_ok=True)
    with open(OUTPUT_CHARS, "w", encoding="utf-8") as f:
        json.dump(characters, f, indent=2, ensure_ascii=False)
    with open(OUTPUT_ALIAS_MAP, "w", encoding="utf-8") as f:
        json.dump(alias_map, f, indent=2, ensure_ascii=False)

    print(f"\n✓ Characters saved to {OUTPUT_CHARS}")
    print(f"✓ Alias map saved to  {OUTPUT_ALIAS_MAP}")

main()

── Step 2: Alias Clustering via Embeddings ──

Loaded 1740 raw names from data\raw_persons.json
After noise filtering: 1601 names
  Loading embedding model 'all-MiniLM-L6-v2' …
  Embedding 1601 names …


Batches:   0%|          | 0/13 [00:00<?, ?it/s]

  Clustering (threshold=0.35) …
  → 949 clusters found

───────────────────────────────────────────────────────
Characters after filtering (>= 10 mentions): 552

Top 30 characters:
   3164  Tyrion                        [Gerion, TYRION, Tyrion Lannister, Tyrion Lannisters (+3 more)]
   2783  Jon                           [JON, JON Jon, Jon Snow, Jon Snows (+2 more)]
   1466  Arya                          [ARYA, Arya Horseface, Arya Stark, Arya Underfoot (+2 more)]
   1365  Jaime                         [JAIME]
   1314  Dany                          [Dancy]
   1115  Stannis                       [Grace King Stannis, King Stannis, King Stanniss, Lord Stannis (+3 more)]
   1103  Catelyn                       [CATELYN, Catelyn Stark, Catelyn Starks, Lady Catelyn (+1 more)]
   1059  Robb                          [Robb Stark, Robb Starks]
    961  Grace                         []
    960  Cersei                        [Cersei Lannister, Cersei Lannisters, Queen Cersei, Queen Cerseis (+1 more

### STEP 3 — Co-occurrence extraction + relationship type classification.

Slides a window over each book and counts how often every pair of canonical characters appears together. Then classifies the dominant relationship type (family / ally / enemy / romance / sworn / other) from the surrounding text. <br>
<br>
Input:  books/*.txt <br>
        data/alias_map.json <br>
        data/canonical_characters.json <br>
Output: data/edges.json   — list of edge dicts

In [3]:
import json
import re
from collections import Counter, defaultdict
from itertools import combinations
from pathlib import Path

# ── Config ────────────────────────────────────────────────────────────────────
BOOKS_DIR      = Path("books")
ALIAS_MAP_FILE = Path("data/alias_map.json")
CHARS_FILE     = Path("data/canonical_characters.json")
OUTPUT_FILE    = Path("data/edges.json")

# How many words to consider a "scene" between two characters
WINDOW_WORDS   = 100

# Minimum co-occurrence count to keep an edge
MIN_WEIGHT     = 5

# ── Relation keywords (extend freely) ────────────────────────────────────────
RELATION_KEYWORDS: dict[str, list[str]] = {
    "family": [
        "father", "mother", "son", "daughter", "brother", "sister",
        "husband", "wife", "uncle", "aunt", "cousin", "nephew", "niece",
        "wed", "wedding", "married", "born", "birth", "bastard", "heir",
        "grandfather", "grandmother", "kin", "blood", "family", "House",
    ],
    "enemy": [
        "killed", "kill", "murder", "murdered", "enemy", "enemies",
        "betrayed", "betrayal", "traitor", "treason", "hated", "hate",
        "fought", "fight", "slew", "slay", "death", "execute", "hanged",
        "war", "battle", "rival", "revenge", "vengeance", "sword",
        "captured", "prisoner", "hostage", "attacked", "threatened",
    ],
    "ally": [
        "ally", "alliance", "allied", "trusted", "trust", "loyal",
        "loyalty", "served", "serve", "helped", "help", "friend",
        "friendship", "together", "joined", "joined forces", "supported",
        "support", "agreement", "deal", "pact", "banner",
    ],
    "romance": [
        "love", "loved", "lover", "kiss", "kissed", "heart", "beloved",
        "desire", "desired", "passion", "bed", "bedded", "wed", "wedded",
        "betrothed", "betrothal", "affection", "longing", "beauty",
        "beautiful", "handsome",
    ],
    "sworn": [
        "sworn sword", "sworn shield", "kingsguard", "queensguard",
        "vow", "oath", "pledged", "pledge", "knelt", "kneel",
        "master", "protector", "bodyguard", "steward", "squire",
        "served", "in service", "commanded", "orders",
    ],
}
# ─────────────────────────────────────────────────────────────────────────────


def load_alias_map(path: Path) -> dict[str, str]:
    with open(path, encoding="utf-8") as f:
        return json.load(f)


def load_characters(path: Path) -> list[dict]:
    with open(path, encoding="utf-8") as f:
        return json.load(f)


def build_regex_map(alias_map: dict[str, str]) -> list[tuple[re.Pattern, str]]:
    """
    Build a list of (compiled regex, canonical_name) pairs.
    Longer aliases first so 'Jon Snow' matches before 'Jon'.
    """
    pairs = sorted(alias_map.items(), key=lambda x: -len(x[0]))
    compiled = []
    for alias, canonical in pairs:
        # Word-boundary match, case-insensitive
        pattern = re.compile(r'\b' + re.escape(alias) + r'\b', re.IGNORECASE)
        compiled.append((pattern, canonical))
    return compiled


def find_chars_in_window(window: str, regex_map) -> set[str]:
    """Return the set of canonical character names found in `window`."""
    found = set()
    for pattern, canonical in regex_map:
        if pattern.search(window):
            found.add(canonical)
    return found


def classify_relation(window: str) -> str:
    """Score each relation type by keyword hits; return the winner."""
    w = window.lower()
    scores: Counter = Counter()
    for rel_type, keywords in RELATION_KEYWORDS.items():
        for kw in keywords:
            if kw in w:
                scores[rel_type] += 1
    if not scores:
        return "other"
    return scores.most_common(1)[0][0]


def extract_edges_from_text(
    text: str,
    regex_map,
    window_words: int = WINDOW_WORDS,
) -> tuple[Counter, dict[tuple, Counter]]:
    """
    Slide a half-overlapping word window over `text`.
    Returns:
      pair_counts : Counter  { (charA, charB) → total co-occurrences }
      pair_types  : dict     { (charA, charB) → Counter of relation types }
    """
    words = text.split()
    step  = window_words // 2

    pair_counts: Counter = Counter()
    pair_types: dict[tuple, Counter] = defaultdict(Counter)

    for i in range(0, len(words) - window_words, step):
        window = " ".join(words[i : i + window_words])
        chars  = find_chars_in_window(window, regex_map)

        if len(chars) < 2:
            continue

        rel = classify_relation(window)

        for a, b in combinations(sorted(chars), 2):
            pair = (a, b)
            pair_counts[pair] += 1
            pair_types[pair][rel] += 1

    return pair_counts, pair_types


def main():
    print("── Step 3: Co-occurrence & Relation Extraction ──\n")

    alias_map  = load_alias_map(ALIAS_MAP_FILE)
    characters = load_characters(CHARS_FILE)
    char_names = {c["name"] for c in characters}

    # Keep only aliases that resolve to a surviving character
    alias_map = {a: c for a, c in alias_map.items() if c in char_names}
    regex_map = build_regex_map(alias_map)
    print(f"Tracking {len(char_names)} characters via {len(alias_map)} aliases\n")

    total_counts: Counter = Counter()
    total_types: dict[tuple, Counter] = defaultdict(Counter)

    for book_path in sorted(Path(BOOKS_DIR).glob("*.txt")):
        print(f"Processing {book_path.name} …")
        text = book_path.read_text(encoding="utf-8", errors="replace")
        counts, types = extract_edges_from_text(text, regex_map)

        total_counts.update(counts)
        for pair, type_counter in types.items():
            total_types[pair].update(type_counter)

        print(f"  → {sum(1 for v in counts.values() if v >= MIN_WEIGHT)} "
              f"edges (>= {MIN_WEIGHT} co-occurrences) in this book")

    # ── Build edge list ───────────────────────────────────────────────────────
    edges = []
    for (source, target), weight in total_counts.items():
        if weight < MIN_WEIGHT:
            continue
        dominant_type = total_types[(source, target)].most_common(1)[0][0]
        type_breakdown = dict(total_types[(source, target)])

        edges.append({
            "source":         source,
            "target":         target,
            "weight":         weight,
            "type":           dominant_type,
            "type_breakdown": type_breakdown,
        })

    # Sort by weight
    edges.sort(key=lambda e: e["weight"], reverse=True)

    # ── Report ────────────────────────────────────────────────────────────────
    print(f"\n{'─'*55}")
    print(f"Total edges (weight >= {MIN_WEIGHT}): {len(edges)}")

    type_counts = Counter(e["type"] for e in edges)
    print("\nEdges by relation type:")
    for rel, cnt in type_counts.most_common():
        print(f"  {rel:<10} {cnt}")

    print("\nTop 20 strongest relationships:")
    for e in edges[:20]:
        print(f"  {e['weight']:5d}  {e['source']:<28} ↔  {e['target']:<28}  [{e['type']}]")

    # ── Save ──────────────────────────────────────────────────────────────────
    with open(OUTPUT_FILE, "w", encoding="utf-8") as f:
        json.dump(edges, f, indent=2, ensure_ascii=False)

    print(f"\n✓ Edges saved to {OUTPUT_FILE}")

main()

── Step 3: Co-occurrence & Relation Extraction ──

Tracking 552 characters via 1174 aliases

Processing GOT1.txt …
  → 2560 edges (>= 5 co-occurrences) in this book
Processing GOT2.txt …
  → 2718 edges (>= 5 co-occurrences) in this book
Processing GOT3.txt …
  → 3952 edges (>= 5 co-occurrences) in this book
Processing GOT4.txt …
  → 2451 edges (>= 5 co-occurrences) in this book
Processing GOT5.txt …
  → 3284 edges (>= 5 co-occurrences) in this book

───────────────────────────────────────────────────────
Total edges (weight >= 5): 12148

Edges by relation type:
  family     11595
  enemy      397
  romance    66
  ally       44
  other      37
  sworn      9

Top 20 strongest relationships:
   1095  Jon                          ↔  Snow                          [family]
    999  Lannister                    ↔  Tyrion                        [family]
    641  Cersei                       ↔  Tyrion                        [family]
    507  Joffrey                      ↔  Sansa Stark        

### STEP 4 — Enrich characters (infer house, compute centrality) and export a single graph JSON ready for D3.js.

Input:  data/canonical_characters.json <br>
        data/edges.json <br>
        books/*.txt <br>
Output: data/graph.json   — { nodes: [...], links: [...] }

In [5]:
import json
import re
from collections import Counter
from pathlib import Path

import networkx as nx

# ── Config ────────────────────────────────────────────────────────────────────
CHARS_FILE   = Path("data/canonical_characters.json")
EDGES_FILE   = Path("data/edges.json")
BOOKS_DIR    = Path("books")
OUTPUT_FILE  = Path("data/graph.json")

# Known house surnames — used for automatic house inference
HOUSE_NAMES = [
    "Stark", "Lannister", "Targaryen", "Baratheon", "Tyrell",
    "Greyjoy", "Martell", "Tully", "Arryn", "Bolton", "Frey",
    "Mormont", "Karstark", "Umber", "Reed", "Manderly",
    "Clegane", "Seaworth", "Dondarrion", "Florent",
]

HOUSE_COLORS = {
    "stark":      "#7a8fa6",
    "lannister":  "#c9a84c",
    "targaryen":  "#b22222",
    "baratheon":  "#6b8c5a",
    "tyrell":     "#4a7c59",
    "greyjoy":    "#3d5a6b",
    "martell":    "#b8620a",
    "tully":      "#2a5fa8",
    "arryn":      "#3a6b8a",
    "bolton":     "#8b2a2a",
    "frey":       "#8a8a6a",
    "mormont":    "#5a7a6a",
    "clegane":    "#6a4a4a",
    "seaworth":   "#4a6a8a",
    "unknown":    "#666688",
}
# ─────────────────────────────────────────────────────────────────────────────


def load_json(path: Path):
    with open(path, encoding="utf-8") as f:
        return json.load(f)


def infer_house_from_name(char_name: str) -> str:
    """
    If the character's canonical name contains a known house surname,
    use that directly (e.g. 'Tyrion Lannister' → 'lannister').
    """
    for house in HOUSE_NAMES:
        if house.lower() in char_name.lower():
            return house.lower()
    return None


def infer_house_from_text(
    char_name: str,
    first_token: str,
    all_text: str,
    window: int = 300,
) -> str:
    """
    Search sentences containing the character's first name;
    count which house word appears most often nearby.
    Returns the best-matching house key or 'unknown'.
    """
    house_counts: Counter = Counter()
    pattern = re.compile(r'\b' + re.escape(first_token) + r'\b', re.IGNORECASE)

    for match in pattern.finditer(all_text):
        start = max(0, match.start() - window)
        end   = min(len(all_text), match.end() + window)
        context = all_text[start:end]
        for house in HOUSE_NAMES:
            if re.search(r'\b' + house + r'\b', context):
                house_counts[house.lower()] += 1

    if house_counts:
        return house_counts.most_common(1)[0][0]
    return "unknown"


def compute_centrality(
    characters: list[dict],
    edges: list[dict],
) -> dict[str, dict]:
    """
    Build a NetworkX graph and compute degree + betweenness centrality.
    Returns { canonical_name → { degree, betweenness } }
    """
    G = nx.Graph()
    for c in characters:
        G.add_node(c["name"])
    for e in edges:
        G.add_edge(e["source"], e["target"], weight=e["weight"])

    degree      = nx.degree_centrality(G)
    betweenness = nx.betweenness_centrality(G, weight="weight", normalized=True)

    return {
        name: {
            "degree":      round(degree.get(name, 0), 4),
            "betweenness": round(betweenness.get(name, 0), 4),
        }
        for name in G.nodes
    }


def main():
    print("── Step 4: Enrich & Export Graph JSON ──\n")

    characters = load_json(CHARS_FILE)
    edges      = load_json(EDGES_FILE)

    # ── Load all book text for house inference ────────────────────────────────
    print("Loading books for house inference …")
    all_text = ""
    for path in sorted(Path(BOOKS_DIR).glob("*.txt")):
        all_text += path.read_text(encoding="utf-8", errors="replace") + "\n"
    print(f"  Total text: {len(all_text):,} chars\n")

    # ── Infer house for each character ────────────────────────────────────────
    print("Inferring houses …")
    for char in characters:
        # 1. Try surname match first (fast & reliable)
        house = infer_house_from_name(char["name"])
        if not house:
            # 2. Fallback: scan surrounding text
            first_token = char["name"].split()[0]
            house = infer_house_from_text(char["name"], first_token, all_text)
        char["house"]  = house
        char["color"]  = HOUSE_COLORS.get(house, HOUSE_COLORS["unknown"])

    # ── Compute graph centrality ───────────────────────────────────────────────
    print("Computing centrality metrics …")
    centrality = compute_centrality(characters, edges)
    for char in characters:
        c = centrality.get(char["name"], {})
        char["degree"]      = c.get("degree", 0)
        char["betweenness"] = c.get("betweenness", 0)

    # ── Build D3-ready structure ───────────────────────────────────────────────
    # D3 force graph expects nodes with unique id and links with source/target
    nodes = [
        {
            "id":          char["name"],      # D3 uses this to match links
            "name":        char["name"],
            "house":       char["house"],
            "color":       char["color"],
            "mentions":    char["mentions"],
            "aliases":     char["aliases"],
            "degree":      char["degree"],
            "betweenness": char["betweenness"],
            # Node size hint for the visualiser (log-scaled)
            "size": max(6, min(30, 6 + char["mentions"] ** 0.45)),
        }
        for char in characters
    ]

    links = [
        {
            "source":         e["source"],
            "target":         e["target"],
            "weight":         e["weight"],
            "type":           e["type"],
            "type_breakdown": e["type_breakdown"],
        }
        for e in edges
    ]

    graph = {"nodes": nodes, "links": links}

    # ── Report ────────────────────────────────────────────────────────────────
    print(f"\n{'─'*55}")
    print(f"Nodes : {len(nodes)}")
    print(f"Links : {len(links)}")

    house_dist = Counter(n["house"] for n in nodes)
    print("\nCharacters by house:")
    for house, cnt in house_dist.most_common():
        print(f"  {house:<15} {cnt}")

    print("\nTop 10 by betweenness centrality (most 'pivotal'):")
    top_bc = sorted(nodes, key=lambda n: n["betweenness"], reverse=True)[:10]
    for n in top_bc:
        print(f"  {n['betweenness']:.4f}  {n['name']}")

    # ── Save ──────────────────────────────────────────────────────────────────
    with open(OUTPUT_FILE, "w", encoding="utf-8") as f:
        json.dump(graph, f, indent=2, ensure_ascii=False)

    print(f"\n✓ Graph saved to {OUTPUT_FILE}")
    print("   → Feed this file into 05_visualise.html")

main()


── Step 4: Enrich & Export Graph JSON ──

Loading books for house inference …
  Total text: 9,437,920 chars

Inferring houses …
Computing centrality metrics …

───────────────────────────────────────────────────────
Nodes : 552
Links : 12148

Characters by house:
  stark           93
  lannister       85
  mormont         79
  greyjoy         38
  tyrell          36
  targaryen       30
  frey            27
  bolton          26
  martell         24
  clegane         18
  unknown         16
  baratheon       15
  arryn           14
  florent         13
  tully           12
  reed            5
  manderly        5
  dondarrion      5
  karstark        5
  seaworth        3
  umber           3

Top 10 by betweenness centrality (most 'pivotal'):
  0.0489  Grey
  0.0383  Fingers
  0.0296  Knights
  0.0278  squires
  0.0272  Grace
  0.0249  Snow
  0.0237  Lannister
  0.0237  Tower
  0.0207  Stark
  0.0204  Winterfell

✓ Graph saved to data\graph.json
   → Feed this file into 05_visualise.html

In [5]:
!python -m http.server 8000

^C


Then open: http://localhost:8000/05_visualise.html